# Exploration
Run `python -m src.build_panel` first (or point `PROCESSED` at `data_demo/processed` after `make demo`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from src.analysis import estimators as est

PROCESSED = pathlib.Path('../data/processed')   # or ../data_demo/processed
panel = pd.read_parquet(PROCESSED / 'panel.parquet')
pev = pd.read_csv(PROCESSED / 'price_events.csv', parse_dates=['date'])
fev = pd.read_csv(PROCESSED / 'free_events.csv', parse_dates=['date'])
panel.describe(include='all').T.head(20)

## Coverage and price provenance

In [ ]:
print(panel.groupby('date').permaslug.nunique().describe())
panel.price_source.value_counts(dropna=False)

## Price-change events

In [ ]:
pev.sort_values('dlogp').head(20)

## Steps 1-2

In [ ]:
f1, f2 = est.cross_model(panel), est.within_model(panel)
pd.concat({'cross': f1.ci(), 'within': f2.ci()})

## Step 3: sensitivity of the event study to window and tier

In [ ]:
rows = []
for W in [7, 14, 21]:
    for tier in [None, 4.0, 2.0]:
        r = est.price_event_study(panel, pev, W=W, tier_ratio=tier)
        if r.get('n_events'):
            rows.append({'W': W, 'tier': tier, 'n_events': r['n_events'],
                         'elasticity': r['dose'].coef['tp_dose'], 'se': r['dose'].se['tp_dose']})
pd.DataFrame(rows)